In [1]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta, timezone
import os

# =========================
# CONFIG
# =========================
SEED = 42
TOTAL_TXNS = 300_000
SUSPICIOUS_PCT = 0.08

N_ENTITIES = 20_000
MAX_ACCTS_PER_ENTITY = 3
N_DAYS = 180
START_DATE = datetime(2025, 1, 1, tzinfo=timezone.utc)

OUTPUT_FOLDER = r"C:\Users\admin\Desktop\DAEN 690\final"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)

CURRENCIES = ["USD", "USD", "USD", "EUR", "GBP"]  # mostly USD

US_STATES = ["CA","NY","TX","FL","IL","WA","VA","MD","NJ","MA","GA","NC","PA","OH","MI"]
COUNTRIES = ["US","US","US","US","CA","MX","GB","DE","IN","CN"]  # mostly US

TXN_TYPES = ["WIRE","ACH","CARD","CASH_DEP","CASH_WD","TRANSFER"]
CHANNELS = ["ONLINE","MOBILE","BRANCH","ATM","POS"]
COUNTERPARTY_TYPES = ["PERSON","BUSINESS","EXCHANGE","UNKNOWN"]

# =========================
# Helper functions
# =========================
def rand_ts():
    day = np.random.randint(0, N_DAYS)
    seconds = np.random.randint(0, 86400)
    return START_DATE + timedelta(days=int(day), seconds=int(seconds))

def log_amount(mean=100, sigma=1.0, min_amt=1, max_amt=50000):
    amt = np.random.lognormal(mean=np.log(mean), sigma=sigma)
    return float(np.clip(amt, min_amt, max_amt))

def make_entity_id(i): return f"E{i:06d}"
def make_account_num(i): return f"A{i:08d}"
def make_counterparty_id(i): return f"C{i:08d}"

def choose_state(): return random.choice(US_STATES)
def choose_country(): return random.choice(COUNTRIES)

# txn_type -> channel distribution
CHANNEL_BY_TYPE = {
    "CARD":      (["POS","ONLINE"], [0.75, 0.25]),
    "ACH":       (["ONLINE","MOBILE"], [0.55, 0.45]),
    "WIRE":      (["ONLINE","BRANCH"], [0.65, 0.35]),
    "TRANSFER":  (["ONLINE","MOBILE"], [0.60, 0.40]),
    "CASH_DEP":  (["ATM","BRANCH"], [0.55, 0.45]),
    "CASH_WD":   (["ATM","BRANCH"], [0.70, 0.30]),
}

# txn_type -> direction default
DEFAULT_DIR = {
    "CASH_DEP": "IN",
    "CASH_WD":  "OUT",
    "CARD":     "OUT",
}

# entity_type -> txn_type weights
TXN_WEIGHTS_PERSON = [0.05, 0.25, 0.45, 0.10, 0.10, 0.05]  # WIRE, ACH, CARD, CASH_DEP, CASH_WD, TRANSFER
TXN_WEIGHTS_BUS    = [0.20, 0.35, 0.10, 0.05, 0.05, 0.25]

# =========================
# 1) Generate entities
# =========================
entities = []
for i in range(1, N_ENTITIES + 1):
    entity_id = make_entity_id(i)
    entity_type = np.random.choice(["PERSON","BUSINESS"], p=[0.85, 0.15])
    home_country = "US" if random.random() < 0.85 else choose_country()
    home_state = choose_state() if home_country == "US" else ""
    risk_tier = np.random.choice(["LOW","MED","HIGH"], p=[0.80, 0.18, 0.02])

    entities.append([entity_id, entity_type, home_country, home_state, risk_tier])

df_entities = pd.DataFrame(entities, columns=["entity_id","entity_type","home_country","home_state","risk_tier"])
df_entities.to_csv(os.path.join(OUTPUT_FOLDER, "entities.csv"), index=False)

# =========================
# 2) Generate accounts (1..3 per entity)
# =========================
accounts = []
acct_list = []
acct_to_entity = {}

acct_counter = 1
for _, row in df_entities.iterrows():
    n_accts = np.random.randint(1, MAX_ACCTS_PER_ENTITY + 1)
    for _ in range(n_accts):
        acc = make_account_num(acct_counter)
        acct_counter += 1
        acct_type = np.random.choice(["CHECKING","SAVINGS","BUSINESS"], p=[0.70, 0.20, 0.10])
        open_dt = (START_DATE - timedelta(days=int(np.random.randint(30, 365*5)))).isoformat()

        accounts.append([acc, row["entity_id"], acct_type, open_dt])
        acct_list.append(acc)
        acct_to_entity[acc] = row["entity_id"]

df_accounts = pd.DataFrame(accounts, columns=["account_num","entity_id","account_type","open_date"])
df_accounts.to_csv(os.path.join(OUTPUT_FOLDER, "accounts.csv"), index=False)

# =========================
# 3) Generate transactions
# =========================
num_suspicious = int(TOTAL_TXNS * SUSPICIOUS_PCT)

transactions = []

# choose a set of suspicious entities to concentrate suspicious behavior (more realistic)
suspicious_entities = set(df_entities.sample(n=max(50, int(N_ENTITIES*0.01)), random_state=SEED)["entity_id"])

for txn_id in range(1, TOTAL_TXNS + 1):
    txn_ts = rand_ts()
    currency = random.choice(CURRENCIES)

    # pick an account
    account_num = random.choice(acct_list)
    entity_id = acct_to_entity[account_num]

    # entity profile drives txn type distribution
    ent_type = df_entities.loc[df_entities["entity_id"] == entity_id, "entity_type"].values[0]
    weights = TXN_WEIGHTS_PERSON if ent_type == "PERSON" else TXN_WEIGHTS_BUS

    txn_type = random.choices(TXN_TYPES, weights=weights, k=1)[0]

    # direction logic
    if txn_type in DEFAULT_DIR:
        direction = DEFAULT_DIR[txn_type]
    else:
        direction = np.random.choice(["IN","OUT"], p=[0.50, 0.50])

    # channel depends on txn_type
    ch_list, ch_w = CHANNEL_BY_TYPE[txn_type]
    channel = random.choices(ch_list, weights=ch_w, k=1)[0]

    # amount depends on type
    if txn_type == "CARD":
        amount = log_amount(mean=40, sigma=0.9, max_amt=3000)
    elif txn_type in ["ACH","TRANSFER"]:
        amount = log_amount(mean=120, sigma=1.0, max_amt=15000)
    elif txn_type == "WIRE":
        amount = log_amount(mean=800, sigma=1.1, max_amt=50000)
    else:  # cash dep / wd
        amount = log_amount(mean=200, sigma=1.0, max_amt=20000)

    # geography
    ent = df_entities[df_entities["entity_id"] == entity_id].iloc[0]
    geo_origin = f"{ent['home_country']}-{ent['home_state']}" if ent["home_country"] == "US" else ent["home_country"]

    # default destination: mostly domestic
    if random.random() < 0.90:
        geo_dest = geo_origin
    else:
        dest_country = choose_country()
        dest_state = choose_state() if dest_country == "US" else ""
        geo_dest = f"{dest_country}-{dest_state}" if dest_country == "US" else dest_country

    # counterparty type
    if txn_type == "CARD":
        counterparty_type = "BUSINESS"
    elif txn_type == "WIRE":
        counterparty_type = random.choices(["BUSINESS","EXCHANGE","UNKNOWN","PERSON"], weights=[0.55,0.15,0.15,0.15], k=1)[0]
    else:
        counterparty_type = random.choices(["PERSON","BUSINESS","UNKNOWN","EXCHANGE"], weights=[0.45,0.40,0.10,0.05], k=1)[0]

    counterparty_id = make_counterparty_id(np.random.randint(1, 200_000))

    # suspicious labeling (simple MVP rule)
    # suspicious if: in suspicious entity set OR within first num_suspicious transactions
    if txn_id <= num_suspicious or entity_id in suspicious_entities:
        is_suspicious = 1 if random.random() < 0.30 else 0  # don't label everything; keeps rate near target
    else:
        is_suspicious = 0

    transactions.append([
        entity_id, txn_id, account_num, txn_ts.isoformat(),
        round(amount,2), currency, txn_type, direction, channel,
        geo_origin, geo_dest, counterparty_type, counterparty_id, is_suspicious
    ])

df_txn = pd.DataFrame(transactions, columns=[
    "entity_id","txn_id","account_num","txn_ts",
    "amount","currency","txn_type","direction","channel",
    "geo_origin","geo_dest","counterparty_type","counterparty_id","is_suspicious"
])

df_txn["txn_ts"] = pd.to_datetime(df_txn["txn_ts"], utc=True)
df_txn = df_txn.sort_values("txn_ts").reset_index(drop=True)
df_txn["txn_id"] = np.arange(1, len(df_txn)+1)

df_txn.to_csv(os.path.join(OUTPUT_FOLDER, "transactions.csv"), index=False)

print("Saved entities.csv, accounts.csv, transactions.csv")
print("Total tx:", len(df_txn))
print("Suspicious %:", round(df_txn["is_suspicious"].mean()*100,2))
print("Txn types:\n", df_txn["txn_type"].value_counts(normalize=True).round(3))
print("Channels:\n", df_txn["channel"].value_counts(normalize=True).round(3))

Saved entities.csv, accounts.csv, transactions.csv
Total tx: 300000
Suspicious %: 2.7
Txn types:
 txn_type
CARD        0.396
ACH         0.265
CASH_WD     0.093
CASH_DEP    0.093
TRANSFER    0.079
WIRE        0.073
Name: proportion, dtype: float64
Channels:
 channel
ONLINE    0.340
POS       0.297
MOBILE    0.152
ATM       0.116
BRANCH    0.095
Name: proportion, dtype: float64


In [3]:
import pandas as pd
import os

# ====== CONFIG: update this to your folder ======
BASE_DIR = r"C:\Users\admin\Desktop\DAEN 690\final"

ENTITIES_FILE = os.path.join(BASE_DIR, "entities.csv")
ACCOUNTS_FILE = os.path.join(BASE_DIR, "accounts.csv")
TXNS_FILE     = os.path.join(BASE_DIR, "transactions.csv")

OUT_FILE      = os.path.join(BASE_DIR, "modeling_dataset.csv")


def main():
    # 1) Read inputs
    entities = pd.read_csv(ENTITIES_FILE)
    accounts = pd.read_csv(ACCOUNTS_FILE)
    txns     = pd.read_csv(TXNS_FILE)

    # 2) Basic type cleanup
    # Ensure IDs are strings (avoid accidental numeric conversion)
    for col in ["entity_id"]:
        entities[col] = entities[col].astype(str)
        accounts[col] = accounts[col].astype(str)
        txns[col]     = txns[col].astype(str)

    accounts["account_num"] = accounts["account_num"].astype(str)
    txns["account_num"]     = txns["account_num"].astype(str)

    # Parse timestamps (keep UTC)
    txns["txn_ts"] = pd.to_datetime(txns["txn_ts"], utc=True, errors="coerce")

    # 3) Merge accounts -> entities (adds entity_type, risk_tier, home geo, etc.)
    acct_entity = accounts.merge(
        entities,
        on="entity_id",
        how="left",
        validate="m:1"  # many accounts per one entity
    )

    # 4) Merge transactions -> acct_entity (adds account + entity context)
    model_df = txns.merge(
        acct_entity,
        on=["account_num", "entity_id"],
        how="left",
        validate="m:1"  # many txns per one account
    )

    # 5) Sort by time and re-create txn_id (optional)
    model_df = model_df.sort_values("txn_ts").reset_index(drop=True)

    # If txn_id already exists and you want to keep it, comment these two lines:
    model_df["txn_id"] = range(1, len(model_df) + 1)

    # 6) Save
    model_df.to_csv(OUT_FILE, index=False)

    # 7) Quick sanity checks
    print("✅ Saved:", OUT_FILE)
    print("Rows:", len(model_df), "Cols:", model_df.shape[1])
    print("Missing entity merges:", model_df["entity_type"].isna().sum())
    print("Missing account merges:", model_df["account_type"].isna().sum())
    print("Suspicious %:", round(model_df["is_suspicious"].mean() * 100, 2))

    print("\nTop txn_type distribution:")
    print(model_df["txn_type"].value_counts(normalize=True).round(3).head(10))

    print("\nTop channel distribution:")
    print(model_df["channel"].value_counts(normalize=True).round(3).head(10))


if __name__ == "__main__":
    main()

✅ Saved: C:\Users\admin\Desktop\DAEN 690\final\modeling_dataset.csv
Rows: 300000 Cols: 20
Missing entity merges: 0
Missing account merges: 0
Suspicious %: 2.7

Top txn_type distribution:
txn_type
CARD        0.396
ACH         0.265
CASH_WD     0.093
CASH_DEP    0.093
TRANSFER    0.079
WIRE        0.073
Name: proportion, dtype: float64

Top channel distribution:
channel
ONLINE    0.340
POS       0.297
MOBILE    0.152
ATM       0.116
BRANCH    0.095
Name: proportion, dtype: float64
